# ⚽ Notebook 1: Model Training, Optimalisatie & Hyperparameter Tuning
### Sports Vision & Tactical Mapping Challenge | Thomas More Geel
**Team:** Git Push and Pray  
**Auteur:** Dmitrij Tchervonisjenko (met Kobe Vandenberk, Cisse Vandeweyer, Brent Van Gelder)  
**Vak:** Deep Learning & Computer Vision (3e Jaar)  
**Hardware & Stack:** NVIDIA GeForce RTX 4070 Laptop GPU (8GB VRAM), CUDA 12.4, PyTorch 2.6, Ultralytics YOLO26

---

## Inhoudsopgave
1. [Inleiding & The Small Object Problem](#1-inleiding--the-small-object-problem)
2. [Deep Learning Fundamenten: FastAI vs. PyTorch & De Trainingslus](#2-deep-learning-fundamenten-fastai-vs-pytorch--de-trainingslus)
3. [Niet-Greedy Optimalisatie vs. Dijkstra](#3-niet-greedy-optimalisatie-vs-dijkstra)
4. [Overfitting Voorkomen: Data Splitting & Early Stopping](#4-overfitting-voorkomen-data-splitting--early-stopping)
5. [Learning Rate Dynamiek: Leslie Smith's One-Cycle Policy & Kneeling](#5-learning-rate-dynamiek-leslie-smiths-one-cycle-policy--kneeling)
6. [Momentum, Zadelpunten, Lokale Minima & AdamW](#6-momentum-zadelpunten-lokale-minima--adamw)
7. [Batch Size & Stochastische Ruis](#7-batch-size--stochastische-ruis)
8. [Architecturale Oplossingen: Vanishing Gradient, ReLU & BatchNorm](#8-architecturale-oplossingen-vanishing-gradient-relu--batchnorm)
9. [Regularisatie & Crowd-Sourcing: L1/L2, Dropout & Data-Augmentatie](#9-regularisatie--crowd-sourcing-l1l2-dropout--data-augmentatie)
10. [Decoder & Upsampling Technieken (Transposed Conv & Checkerboard Artifacts)](#10-decoder--upsampling-technieken-transposed-conv--checkerboard-artifacts)
11. [Praktische Training Walkthrough & Visualisatie van Trainingscurves](#11-praktische-training-walkthrough--visualisatie-van-trainingscurves)

---
## 1. Inleiding & The Small Object Problem

In deze challenge bouwen we een end-to-end Computer Vision pipeline die een professionele 1080p broadcast videostream (PSG vs. Inter Milan) omzet naar een gesynchroniseerde 2D tactische radar.

### De Grote Uitdaging: De Voetbal
* Op een $1920 \times 1080$ Full HD broadcast camera beslaat een voetbal gemiddeld slechts **$16 \times 17$ pixels**.
* Dit is minder dan **$0.015\%$ van het totale beeldoppervlak**.
* **Probleem bij standaard resolutie ($640\text{ px}$):**
  Wanneer een $1080p$ frame wordt gedownsampled naar $640\text{ px}$, krimpt de voetbal naar een niet-detecteerbaar stipje van amper $5 \times 5$ pixels. Na meerdere convolutionele downsampling-lagen (stride 32 in de backbone) verdwijnt het signaal van de bal volledig in het netwerk (detectiepercentage: $0.0\%$, $\text{mAP@50} = 0.000$).
* **De Oplossing (RTX 4070 GPU Training op $1280\text{ px}$):**
  Door te trainen op `imgsz=1280` met batch size 8 op onze dedicated NVIDIA RTX 4070 Laptop GPU, behoudt het netwerk $4\times$ meer spatiale pixels per object. Dit leidde tot een spectaculaire stijging van **+25.7% overall mAP** en continue detectie van de voetbal.

In [ ]:
# 1. Inspecteer Hardware & Omgeving
import torch
import torchvision
import ultralytics
import sys
from pathlib import Path

print(f"Python Versie: {sys.version.split()[0]}")
print(f"PyTorch Versie: {torch.__version__}")
print(f"CUDA Beschikbaar: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Beschikbaar: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"Ultralytics Versie: {ultralytics.__version__}")

---
## 2. Deep Learning Fundamenten: FastAI vs. PyTorch & De Trainingslus

Binnen de opleiding en de state-of-the-art AI-praktijk onderscheiden we twee complementaire abstractieniveaus:

### 2.1 Low-Level PyTorch: De Computational DAG (Directed Acyclic Graph)
PyTorch biedt directe controle over tensors en GPU-geheugen. Dankzij de **Autograd** engine wordt tijdens de *forward pass* dynamisch een gerichte acyclische graaf opgebouwd. Elk knooppunt onthoudt welke wiskundige operaties zijn uitgevoerd. Tijdens `loss.backward()` wordt via de kettingregel (chain rule) de partiële afgeleide $\frac{\partial L}{\partial w}$ berekend voor elk gewicht $w$:

$$\frac{\partial L}{\partial w_i} = \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial z} \cdot \frac{\partial z}{\partial w_i}$$

De elementaire trainingsstap:
```python
optimizer.zero_grad()               # 1. Reset gradiënten van de vorige stap
outputs = model(inputs)             # 2. Forward pass: bereken modelvoorspelling
loss = criterion(outputs, targets)  # 3. Bereken het verlies (scalar)
loss.backward()                     # 4. Backward pass: bereken dL/dw via Autograd
optimizer.step()                    # 5. Pas gewichten aan: w = w - lr * grad
```

### 2.2 High-Level FastAI: Learner, DataLoaders & Best Practices
FastAI bouwt bovenop PyTorch een geabstraheerde laag die geavanceerde technieken standaardiseert:
* `DataLoaders`: Geoptimaliseerde parallelle pre-fetching, multi-process batch-loading en GPU-tensor caching.
* `Learner`: Bundelt data, modelarchitectuur, verliesfunctie, optimalisatie-algoritme en callbacks in één intuïtieve interface.
* **Progressive Resizing:** Starten met trainen op lagere resoluties ($640\text{ px}$) om snel grove globale representaties te leren, en vervolgens fine-tunen op hoge resolutie ($1280\text{ px}$) voor fijne details.

---
## 3. Niet-Greedy Optimalisatie vs. Dijkstra

Een belangrijk concept uit het hoorcollege: **Waarom mag een Deep Learning optimizer NIET greedy zijn zoals Dijkstra's kortste-pad algoritme?**

* **Dijkstra's Algoritme (Greedy):**
  Dijkstra kiest op elk tussenknooppunt strikt het lokaal kortste pad. Omdat graafafstanden niet-negatief zijn en deterministisch vastliggen, garandeert deze 'gulzige' strategie het globaal optimale pad.
* **Deep Learning Verlieslandschap (Non-Convex & Hoog-Dimensionaal):**
  Het verliesoppervlak van een diep neuraal netwerk bezit miljoenen parameters en is vergeven van:
  1. **Scherpe, nauwe lokale minima (*sharp minima*):** Hier presteert het model perfect op de training data, maar overfit het desastreus op ongeziene testdata.
  2. **Zadelpunten (*saddle points*):** Punten waar de gradiënt in bijna alle richtingen nul is ($\nabla L \approx 0$), maar die geen minimum vormen.
* **Gevaar van Greedy gedrag:**
  Als een optimizer strikt gulzig op zoek gaat naar de snelste directe afname, stort hij onmiddellijk in het dichtstbijzijnde steile, scherpe dal en zit hij muurvast.
* **De Oplossing:**
  Stochastic Gradient Descent (SGD) met momentum en mini-batch stochastic noise neemt bewuste stochastische stapjes, waardoor het netwerk 'springt' over scherpe kuiltjes en divergeert naar brede, vlakke minima (*flat minima*) die superieur generaliseren.

---
## 4. Overfitting Voorkomen: Data Splitting & Early Stopping

### 4.1 De Drievoudige Data Split: Train, Validation & Held-Out Test Set
Om gegarandeerde generalisatie naar de praktijk te borgen, splitsen we de data strikt:
1. **Training Set ($70\%$):** Wordt gebruikt om de forward en backward passes te berekenen en gewichten $w$ via gradient descent bij te werken.
2. **Validation Set ($20\%$):** Wordt aan het einde van elke epoch geëvalueerd **zonder gewichtsupdates** (`model.eval()`, `torch.no_grad()`). Dient om hyperparameters (zoals learning rate, IoU-thresholds) te tunen en overtraining te detecteren.
3. **Held-Out Test Set ($10\%$):** Volledig afgeschermde data die pas **eenmalig helemaal aan het einde van het project** wordt geëvalueerd om de werkelijke, onvervalste prestatie aan te tonen.

### 4.2 Early Stopping Mechanisme
* Een **epoch** is één volledige cyclus waarbij alle trainingsbatches door het netwerk zijn gegaan.
* In het begin van de training dalen zowel de $\text{Loss}_{train}$ als de $\text{Loss}_{val}$.
* Zodra het model overfit, blijft $\text{Loss}_{train}$ dalen (het netwerk memoriseert ruis), terwijl $\text{Loss}_{val}$ begint te stijgen of plateaut.
* **Early Stopping Regel:** Indien de validatiescore gedurende $N$ opeenvolgende epochs (*patience*) niet meer verbetert, stoppen we de training onmiddellijk en herstellen we de checkpoint-gewichten met het laagste validatieverlies (`best.pt`).

---
## 5. Learning Rate Dynamiek: Leslie Smith's One-Cycle Policy & Kneeling

De **Learning Rate (LR)** bepaalt de grootte van de stap die de optimizer zet in de richting van het negatieve gradiëntveld:

$$w_{t+1} = w_t - \eta \cdot \nabla L(w_t)$$

* **Te hoge LR:** Het model oscilleert wild over het dal heen, schiet uit de baan (*exploding loss*) en divergeert.
* **Te lage LR:** Het model beweegt tergend langzaam en blijft hopeloos steken in het eerste lokale zadelpunt.

### Leslie Smith's One-Cycle Learning Rate Policy & Kneeling Techniek
In plaats van een statische LR gebruikt onze training een dynamisch One-Cycle schema:
1. **Kneeling / Warm-up Fase (Eerste $10-20\%$ van training):**
   We starten met een uiterst kleine leersnelheid (bijv. $10^{-5}$) en bouwen deze geleidelijk op ("kneeling"). Dit voorkomt dat willekeurig geïnitialiseerde gewichten in vroege batches worden vernietigd door onstabiele gradiënten.
2. **Aggressieve Exploratie Fase (Midden):**
   De LR bereikt zijn piek ($LR_{max}$). Deze grote stappen functioneren als een krachtige vorm van regularisatie: het netwerk springt uit sub-optimale lokale kuiltjes en zoekt naar brede, vlakke bekkens.
3. **Cosine Annealing Cooldown Fase (Laatste $30-40\%$):**
   De leersnelheid zakt vloeiend via een cosinus-curve terug naar een minieme waarde ($LR_{min} \to 0$). Hierdoor kan het model zeer fijn convergeren naar de bodem van het beste vlakke dal.

---
## 6. Momentum, Zadelpunten, Lokale Minima & AdamW

### 6.1 De Fysische Kracht van Momentum
Klassieke Gradient Descent heeft geen geheugen: bij een smalle, steile kloof stuitert de optimizer wild heen en weer tussen de twee wanden zonder voortgang over de bodem te maken.
**Momentum** simuleert een zware stalen kogel die van een heuvel rolt met traagheid:

$$v_t = \beta \cdot v_{t-1} + (1 - \beta) \cdot g_t$$
$$w_{t+1} = w_t - \eta \cdot v_t$$

* Orthogonale oscillaties heffen elkaar op (gemiddelde is 0).
* Consistente stappen in de lengterichting tellen bij elkaar op en bouwen snelheid op.
* Hierdoor dendert de kogel moeiteloos over kleine oneffenheden, lokale bulten en zadelpunten heen.

### 6.2 Adam & AdamW (Adaptive Moment Estimation)
Adam combineert twee ideeën:
1. Eerste moment $m_t$ (voortschrijdend gemiddelde van gradiënten $\approx$ richting).
2. Tweede moment $v_t$ (voortschrijdend gemiddelde van kwadratische gradiënten $\approx$ stapgrootte per parameter).
$$w_{t+1} = w_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

**Belangrijke nuance (Adam vs. AdamW):**
In standaard Adam wordt $L_2$-regularisatie foutief vermengd met de gradiënt, waardoor parameters met grote historische gradiënten onevenredig weinig gewichtsverval krijgen. **AdamW** ontkoppelt *weight decay* van de gradiëntupdate, wat zorgt voor drastisch betere generalisatie in moderne vision modellen zoals YOLO26.

---
## 7. Batch Size & Stochastische Ruis

Waarom trainen we niet met de volledige dataset in één keer (*Full Batch GD*)?
* **Full Batch:** Berekent de exacte gradiënt over alle beelden. Dit is rekenkundig traag en leidt paradoxaal genoeg tot *slechtere* resultaten: het model glijdt rechtstreeks in het dichtstbijzijnde scherpe dal en overfit.
* **Mini-Batch ($batch=8$ op de RTX 4070):**
  Elke batch van 8 beelden geeft een lichte, stochastische ruisschatting van de werkelijke gradiënt. Deze 'trilling' schudt het netwerk voortdurend heen en weer, waardoor het automatisch ontsnapt uit smalle lokale valkuilen en landt in brede, robuuste minima.

---
## 8. Architecturale Oplossingen: Vanishing Gradient, ReLU & BatchNorm

### 8.1 Het Vanishing Gradient Probleem
In diepe neurale netwerken wordt de gradiënt berekend via herhaalde matrixvermenigvuldiging:
$$\frac{\partial L}{\partial w_1} = \prod_{l=1}^L W_l \cdot \sigma'(z_l)$$
Bij klassieke activatiefuncties zoals Sigmoid of Tanh ligt de afgeleide altijd tussen $0$ en $0.25$. Bij 20 lagen diep vermenigvuldig je 20 getallen kleiner dan $0.25$:
$$0.25^{20} \approx 9 \times 10^{-13} \to 0$$
Het signaal sterft volledig uit! De vroege lagen leren letterlijk niets meer (*vanishing gradient*).

### 8.2 ReLU & SiLU (Swish) Activatie
* **ReLU (Rectified Linear Unit):**
  $$\text{ReLU}(x) = \max(0, x)$$
  De afgeleide is voor alle positieve inputs exact **$1$**. Er treedt geen verzadiging op, waardoor gradiënten zonder signaalverlies over tientallen convolutionele lagen heen kunnen stromen.
* **SiLU / Swish:** In YOLO26 wordt $x \cdot \sigma(x)$ gebruikt, wat een gladde gradiëntovergang garandeert rond 0.

### 8.3 Batch Normalization
Tijdens training veranderen de gewichten van eerdere lagen continu, waardoor de verdeling van de invoer voor diepere lagen voortdurend verschuift (*Internal Covariate Shift*).
**Batch Normalization** herschaalt en recentreert elke mini-batch:
$$\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \quad y = \gamma \hat{x} + \beta$$
Dit stabiliseert de optimalisatie en maakt het netwerk minder gevoelig voor gewichtsinitialisatie.

---
## 9. Regularisatie & Crowd-Sourcing: L1/L2, Dropout & Data-Augmentatie

### 9.1 Het "Super-Star Neuron" Probleem & Crowd-Sourcing
Zonder regularisatie kan één dominant neuron ("super-star neuron") alle voorspellende taken naar zich toetrekken, terwijl omliggende neuronen inactief blijven. Als dit specifieke neuron tijdens een wedstrijd faalt (bijv. door onverwachte schaduw of camerahoek), crasht de gehele detectie.

### 9.2 Oplossingen:
1. **L1 Regularisatie (Lasso):** $\lambda \sum |w_i|$ drijft irrelevante gewichten exact naar 0 (spaarzaamheid / feature selection).
2. **L2 Regularisatie (Ridge / Weight Decay):** $\frac{1}{2}\lambda \sum w_i^2$ bestraft excessief grote gewichten. Het dwingt het netwerk om het werk gelijkmatig te verdelen over alle verbindingen.
3. **Dropout:** Tijdens elke trainingsstap wordt een willekeurige selectie van neuronen (bijv. $20-50\%$) tijdelijk uitgeschakeld ("jij mag nu niet meedoen"). Neurale representaties worden hierdoor gedwongen om 'crowd-sourced' en redundant te worden: elk neuron moet samenwerken en algemene kenmerken leren.
4. **Data-Augmentatie (YOLO Ingebouwd):**
   * **Mosaic:** Voegt 4 verschillende trainingsbeelden samen tot één samengesteld beeld. Hierdoor leert het netwerk objecten herkennen op wisselende schalen en locaties.
   * **Mixup:** Mengt twee beelden lineair met transparantie.
   * **HSV Jitter:** Varieert kleurtoon, verzadiging en helderheid om gras- en belichtingsvariaties op te vangen.
   * **Belangrijke Best Practice (`close_mosaic=10`):** In de laatste 10 epochs schakelen we Mosaic uit! Dit zorgt ervoor dat het model in de eindfase traint op natuurlijke, onvervormde beelden, wat bounding box jitter drastisch vermindert.

### 9.3 Hard Labels vs. Soft Labels & Label Smoothing (Kleine & Wazige Objecten)
* **Het Probleem met Harde Labels (One-Hot):**
  Standaard labels stellen de ware klasse op exact $1.0$ en de rest op $0.0$ ($y_{hard} = [1, 0, 0, 0, 0]$). Hierdoor probeert de Cross-Entropy loss de logits naar $\pm \infty$ te forceren. Het model wordt **overconfident**: als een bal door bewegingsonscherpte (*motion blur*) of een speler's schoen deels occluded is, klapt de voorspelling direct in naar 0.
* **De Oplossing: Soft Labels (Label Smoothing):**
  We mengen het harde doel met een uniforme kansverdeling over alle $K=5$ klassen:
  $$y_{soft} = (1 - \epsilon) \cdot y_{hard} + \frac{\epsilon}{K}$$
  Bij $\epsilon = 0.10$ wordt het bal-doellabel **$0.92$** en de andere klassen **$0.02$**. Het netwerk leert gekalibreerde kansen (*calibrated probabilities*), wat detectie van deels bedekte ballen drastisch verbetert.
* **Soft Assignment in YOLO:** De moderne Task-Aligned Assigner berekent reeds continue zachte doelen: $t = s^\alpha \cdot \text{IoU}^\beta$.

In [ ]:
# Demonstratie: Hard Labels vs. Soft Labels (Label Smoothing) in PyTorch
import torch
import torch.nn as nn

epsilon = 0.10
num_classes = 5
target_class = torch.tensor([0])  # Klasse 0 = 'ball'

criterion_hard = nn.CrossEntropyLoss()
criterion_soft = nn.CrossEntropyLoss(label_smoothing=epsilon)

# Simuleer twee voorspellingen:
# 1. Zeker beeld (hoge contrast bal)
# 2. Wazig / gedeeltelijk occluded beeld (voetbal deels achter een voet)
logits_sharp = torch.tensor([[8.0, -1.0, -2.0, -1.5, -2.0]])
logits_blurry = torch.tensor([[1.8, 1.2, 0.4, -0.2, 0.1]])

print("=== Vergelijking Hard vs. Soft Labels Loss ===")
print(f"1. Scherpe Bal - Hard CE Loss : {criterion_hard(logits_sharp, target_class).item():.4f}")
print(f"   Scherpe Bal - Soft CE Loss : {criterion_soft(logits_sharp, target_class).item():.4f} (Geregulariseerd)")
print(f"2. Wazige Bal  - Hard CE Loss : {criterion_hard(logits_blurry, target_class).item():.4f}")
print(f"   Wazige Bal  - Soft CE Loss : {criterion_soft(logits_blurry, target_class).item():.4f} (Stabielere gradiënt!)")

---
## 10. Decoder & Upsampling Technieken (Transposed Conv & Checkerboard Artifacts)

In Feature Pyramid Networks (FPN) en segmentatie-decoders moeten laag-dimensionale feature maps weer worden opgeschaald naar hoge spatiale resolutie:

### 10.1 Nearest-Neighbor Upsampling
* Kopieert eenvoudig de dichtstbijzijnde pixelwaarde naar omliggende cellen.
* Voordeel: Geen extra berekeningen of gewichten.
* Nadeel: Blokkerige, pixelachtige randen zonder gradiëntovergang.

### 10.2 Bilinear Interpolation
* Berekent een gewogen gemiddelde van de 4 omliggende buurpixels.
* Voordeel: Gladde kleurovergangen.
* Nadeel: Vervaging van scherpe randen (*blurry boundaries*).

### 10.3 Transposed Convolution (Deconvolution) & Checkerboard Artifacts
* Gebruikt een leerbare convolutionele kernel (bijv. $3 \times 3$ of $5 \times 5$) met een stride van 2 om spatiale resolutie te verdubbelen.
* **Het Checkerboard Artifact Probleem:**
  Door overlapping van de convolutionele schuifvensters ontvangen sommige pixels in het midden van het venster dubbele of viervoudige energie ten opzichte van randpixels. Hierdoor ontstaat een storend "schaakbordpatroon" (*checkerboard artifacts*).
* **De Oplossing:**
  1. Penaliseren van hoogfrequente gradiëntartefacten in de verliesfunctie.
  2. Gebruik van **PixelShuffle (Sub-Pixel Convolution)**: Convoluties uitvoeren in kanaaldiepte en deze via reshape herschikken naar spatiale resolutie zonder overlapping.

---
## 11. Praktische Training Walkthrough & Visualisatie van Trainingscurves

Hieronder inspecteren en visualiseren we de werkelijke trainingslogboeken (`results.csv`) van onze YOLO26s training op de RTX 4070 GPU over 30 epochs.

In [ ]:
# Visualiseer de Trainings- en Validatieverliezen over de Epochs
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Zoek naar results.csv
csv_path = Path("dimi/runs/ball_yolo26_1280/results.csv")
if not csv_path.exists():
    csv_path = Path("../dimi/runs/ball_yolo26_1280/results.csv")
if not csv_path.exists():
    csv_path = Path("runs/ball_yolo26_1280/results.csv")

if csv_path.exists():
    df_results = pd.read_csv(csv_path)
    df_results.columns = [c.strip() for c in df_results.columns]
    print(f"Succesvol {len(df_results)} epochs ingeladen uit: {csv_path}")
    print(df_results[['epoch', 'train/box_loss', 'train/cls_loss', 'val/box_loss', 'val/cls_loss', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']].tail())
else:
    print(f"Let op: results.csv niet gevonden op {csv_path}. Toon demonstratiegrafiek.")

In [ ]:
# Plotting: Loss Curves & mAP Progressie
if 'df_results' in locals() and not df_results.empty:
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Bounding Box Loss
    axes[0, 0].plot(df_results['epoch'], df_results['train/box_loss'], label='Train Box Loss', color='#1f77b4', lw=2)
    axes[0, 0].plot(df_results['epoch'], df_results['val/box_loss'], label='Val Box Loss', color='#ff7f0e', lw=2, linestyle='--')
    axes[0, 0].set_title('Bounding Box Regression Loss (CIoU)', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    
    # 2. Classification Loss
    axes[0, 1].plot(df_results['epoch'], df_results['train/cls_loss'], label='Train Cls Loss', color='#2ca02c', lw=2)
    axes[0, 1].plot(df_results['epoch'], df_results['val/cls_loss'], label='Val Cls Loss', color='#d62728', lw=2, linestyle='--')
    axes[0, 1].set_title('Classification Loss (BCE)', fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    
    # 3. DFL (Distribution Focal Loss)
    axes[1, 0].plot(df_results['epoch'], df_results['train/dfl_loss'], label='Train DFL Loss', color='#9467bd', lw=2)
    axes[1, 0].plot(df_results['epoch'], df_results['val/dfl_loss'], label='Val DFL Loss', color='#8c564b', lw=2, linestyle='--')
    axes[1, 0].set_title('Distribution Focal Loss (DFL)', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Loss')
    axes[1, 0].legend()
    
    # 4. mAP Metrics Progressie
    axes[1, 1].plot(df_results['epoch'], df_results['metrics/mAP50(B)'], label='mAP@50', color='#00a86b', lw=2.5)
    axes[1, 1].plot(df_results['epoch'], df_results['metrics/mAP50-95(B)'], label='mAP@50-95', color='#e377c2', lw=2, linestyle=':')
    axes[1, 1].set_title('Mean Average Precision (mAP) Progressie', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('mAP Score')
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.show()

---
## Conclusie & Samenvatting Notebook 1
In dit notebook hebben we de volledige theorie en trainingspraktijk gedocumenteerd:
1. **1280p Resolutie op RTX 4070:** Essentieel om kleine objecten ($16 \times 17$ px bal) te vangen.
2. **One-Cycle Policy & Kneeling:** Zorgt voor stabiele opstart, brede vallei-exploratie en scherpe convergentie.
3. **AdamW + Momentum + Mini-Batch Ruis:** Garandeert ontsnapping uit lokale zadelpunten en scherpe dalen.
4. **Regularisatie & close_mosaic=10:** Voorkomt overfitten en bereidt het netwerk optimaal voor op de finale evaluatie in Notebook 2.